# Familiar Effect-Size Interpretation — Complete Solution

Fully worked interpretation of Cohen’s *d*, Cramér’s *V* and η² for the Familiar datasets, including benchmarks, bootstrap CIs, practical translations, audience language, sensitivity simulation and alternate implementations.


## Flowchart: Effect-Size Interpretation Pipeline

Statistical significance alone is not enough for Familiar’s product decisions.  
This flowchart shows how to move from a p-value to a *practically interpretable* effect size that different audiences can act on.

```mermaid
flowchart TD
    A[Start: Significant result?<br/>t-test / Chi-square / ANOVA] --> B[Compute effect size<br/>Cohen d / Cramér V / η²]
    B --> C[Obtain uncertainty<br/>Bootstrap or analytic CI<br/>for the effect size]
    C --> D[Apply conventional benchmarks<br/>Cohen / Cohen-Sawilowsky<br/>or field-specific norms]
    D --> E{Audience?}
    E -->|Executive| F[Translate to plain language<br/>"about 1.3 extra years"<br/>"70 % of Vein users low iron"]
    E -->|Technical| G[Report d / V / η² + CI<br/>+ assumption checks]
    E -->|Mixed| H[Layered report:<br/>headline → magnitude → diagnostics]
    F --> I[Business decision<br/>Marketing claim / counselling / next study]
    G --> I
    H --> I
    I --> J[Simulation / sensitivity<br/>How does interpretation change<br/>if N or SD changes?]
    style D fill:#fff3cd,stroke:#856404
    style I fill:#e6f3ff,stroke:#0066cc
    style J fill:#e8f5e9,stroke:#2e7d32
```

**Key point from the audience PDFs:** Executives need the *practical* magnitude; technical reviewers need the coefficient and its confidence interval.  Both must appear in the final write-up.


## Audience Considerations (from the supplied PDFs)

1. **Data literacy** (Jočys)  
   - High-literacy readers understand “Cohen’s d = 0.62 (medium)”.  
   - Lower-literacy readers need a concrete translation: “Vein subscribers live roughly 1.3 years longer on average – a difference most people would notice.”

2. **Audience type** (McMurrey)  
   - Executives care about risk, ROI and claim strength.  
   - Technical supervisors want the formula, CI and assumption checks.  
   - Nonspecialists (board / potential customers) need plain-language side-effect guidance.

3. **Report structure** (paper-structure.pdf)  
   - Headlines + practical magnitude in Introduction / Conclusion.  
   - Full effect-size calculations, CIs and simulation in Body / Appendix.

For Familiar the primary readers are product leadership (executive) and the medical-advisory board (technical).  The notebooks therefore always pair the numeric effect size with a one-sentence plain-language gloss.


## 1. Setup & recompute the three effect sizes


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind, chi2_contingency, f_oneway
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

lifespans = pd.read_csv("familiar_lifespan.csv")
iron = pd.read_csv("familiar_iron.csv")

# --- Cohen's d (Vein - Artery lifespan) ---
vein = lifespans.loc[lifespans.pack == "vein", "lifespan"].to_numpy()
artery = lifespans.loc[lifespans.pack == "artery", "lifespan"].to_numpy()
nx, ny = len(vein), len(artery)
pooled_sd = np.sqrt(((nx-1)*np.var(vein, ddof=1) + (ny-1)*np.var(artery, ddof=1)) / (nx+ny-2))
d = (vein.mean() - artery.mean()) / pooled_sd

# --- Cramér's V ---
Xtab = pd.crosstab(iron.pack, iron.iron)
chi2, p_chi, dof, exp = chi2_contingency(Xtab)
n_total = Xtab.values.sum()
V = np.sqrt(chi2 / (n_total * (min(Xtab.shape) - 1)))

# --- η² from ordinal iron score ---
mapping = {"low": 1, "normal": 2, "high": 3}
iron = iron.copy()
iron["score"] = iron["iron"].map(mapping)
grand_mean = iron["score"].mean()
ss_total = ((iron["score"] - grand_mean)**2).sum()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2
                 for _, g in iron.groupby("pack")["score"])
eta2 = ss_between / ss_total

print(f"Cohen's d (lifespan Vein-Artery) = {d:.3f}")
print(f"Cramér's V (pack × iron)         = {V:.3f}")
print(f"η² (ordinal iron ANOVA)          = {eta2:.3f}")
print(f"\nMean lifespan difference (years) = {vein.mean()-artery.mean():.3f}")
print(f"Mean score difference (Artery-Vein) = {iron.loc[iron.pack=='artery','score'].mean()-iron.loc[iron.pack=='vein','score'].mean():.3f}")


## 2. Conventional benchmarks & plain-language glosses


In [ ]:
def classify_d(x):
    ax = abs(x)
    if ax < 0.2: return "negligible"
    if ax < 0.5: return "small"
    if ax < 0.8: return "medium"
    return "large"

def classify_V(x):
    # common rule of thumb for tables with df* = min(r-1,c-1) = 1 or 2
    if x < 0.1: return "negligible"
    if x < 0.3: return "small"
    if x < 0.5: return "medium"
    return "large"

def classify_eta(x):
    if x < 0.01: return "negligible"
    if x < 0.06: return "small"
    if x < 0.14: return "medium"
    return "large"

print("=== Classification ===")
print(f"Cohen's d = {d:.3f}  →  {classify_d(d)}")
print(f"Cramér V  = {V:.3f}  →  {classify_V(V)}")
print(f"η²        = {eta2:.3f}  →  {classify_eta(eta2)}")

print("\n=== Executive one-liners ===")
print(f"Lifespan: Vein subscribers live ~{vein.mean()-artery.mean():.1f} years longer on average "
      f"(standardised effect d = {d:.2f}, medium).")
print(f"Iron: 70 % of Vein users have low iron vs only 20 % of Artery users "
      f"(Cramér V = {V:.2f}, large association).")
print(f"Ordinal iron score shifts by a full point (η² = {eta2:.2f}, large); "
      f"the packs produce clearly different iron profiles.")


## 3. Bootstrap confidence intervals for the effect sizes


In [ ]:
np.random.seed(42)
n_boot = 2000

# --- Bootstrap Cohen's d ---
boot_d = np.empty(n_boot)
for i in range(n_boot):
    bv = np.random.choice(vein, size=nx, replace=True)
    ba = np.random.choice(artery, size=ny, replace=True)
    ps = np.sqrt(((nx-1)*np.var(bv, ddof=1) + (ny-1)*np.var(ba, ddof=1)) / (nx+ny-2))
    boot_d[i] = (bv.mean() - ba.mean()) / ps
ci_d = np.percentile(boot_d, [2.5, 97.5])

# --- Bootstrap Cramér's V (resample rows) ---
boot_V = np.empty(n_boot)
iron_arr = iron[["pack", "iron"]].to_numpy()
for i in range(n_boot):
    idx = np.random.randint(0, len(iron_arr), size=len(iron_arr))
    sample = iron_arr[idx]
    ct = pd.crosstab(sample[:,0], sample[:,1])
    # guard against missing categories in a given resample
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        boot_V[i] = np.nan
        continue
    c2 = chi2_contingency(ct)[0]
    boot_V[i] = np.sqrt(c2 / (ct.values.sum() * (min(ct.shape)-1)))
ci_V = np.nanpercentile(boot_V, [2.5, 97.5])

# --- Bootstrap η² ---
boot_eta = np.empty(n_boot)
scores = iron["score"].to_numpy()
packs = iron["pack"].to_numpy()
for i in range(n_boot):
    idx = np.random.randint(0, len(scores), size=len(scores))
    s, p = scores[idx], packs[idx]
    gm = s.mean()
    sst = ((s-gm)**2).sum()
    ssb = 0.0
    for lab in np.unique(p):
        g = s[p==lab]
        ssb += len(g) * (g.mean()-gm)**2
    boot_eta[i] = ssb / sst if sst > 0 else np.nan
ci_eta = np.nanpercentile(boot_eta, [2.5, 97.5])

print(f"Cohen's d  95% bootstrap CI: [{ci_d[0]:.3f}, {ci_d[1]:.3f}]")
print(f"  includes 0? {ci_d[0] < 0 < ci_d[1]}   includes 0.5 (medium)? {ci_d[0] < 0.5 < ci_d[1]}")
print(f"Cramér V   95% bootstrap CI: [{ci_V[0]:.3f}, {ci_V[1]:.3f}]")
print(f"η²         95% bootstrap CI: [{ci_eta[0]:.3f}, {ci_eta[1]:.3f}]")


## 4. Practical magnitude vs statistical magnitude


In [ ]:
diff_years = vein.mean() - artery.mean()
print(f"Raw mean difference (years)     = {diff_years:.3f}")
print(f"Standardised (Cohen's d)        = {d:.3f}")
print("→ For executives put the raw years first; for technical peers also report d + CI.")

ct = pd.crosstab(iron.pack, iron.iron)
prop = ct.div(ct.sum(axis=1), axis=0)
low_vein = prop.loc["vein", "low"]
low_artery = prop.loc["artery", "low"]
print(f"\n% low iron – Vein   = {100*low_vein:.1f}%")
print(f"% low iron – Artery = {100*low_artery:.1f}%")
print(f"Percentage-point gap = {100*(low_vein-low_artery):.1f} pp")
print("→ A 50-percentage-point gap in ‘low iron’ is immediately actionable for counselling.")


## 5. Visualization – effect sizes with CIs and benchmarks


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

labels = ["Cohen's d\n(lifespan)", "Cramér's V\n(iron cat.)", "η²\n(iron ordinal)"]
points = [d, V, eta2]
los = [ci_d[0], ci_V[0], ci_eta[0]]
his = [ci_d[1], ci_V[1], ci_eta[1]]
colors = ["#4c72b0", "#dd8452", "#55a868"]

for i, (lab, pt, lo, hi, c) in enumerate(zip(labels, points, los, his, colors)):
    ax.errorbar(pt, i, xerr=[[pt-lo], [hi-pt]], fmt="o", color=c, capsize=6, markersize=9, label=lab)
    ax.text(hi+0.02, i, f"{pt:.2f}", va="center", fontsize=10)

# benchmark lines (approximate “medium” thresholds)
ax.axvline(0.5, color="gray", ls="--", lw=1, alpha=0.7, label="d / V medium ≈ 0.5")
ax.axvline(0.14, color="gray", ls=":", lw=1, alpha=0.7, label="η² medium ≈ 0.14")

ax.set_yticks(range(3))
ax.set_yticklabels(labels)
ax.set_xlabel("Effect-size value (with 95 % bootstrap CI)")
ax.set_title("Familiar Effect Sizes – Magnitude & Uncertainty")
ax.set_xlim(-0.1, 1.0)
ax.legend(loc="lower right", fontsize=8)
ax.axvline(0, color="black", lw=0.8)
plt.tight_layout()
plt.savefig("familiar_effect_size_visuals.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved as familiar_effect_size_visuals.png")


## 6. Simulation – sample-size & variability sensitivity


In [ ]:
np.random.seed(7)
print("=== Sample-size sensitivity for Cohen's d ===")
for n_sub in [8, 12, 16, 20]:
    # draw without replacement from the observed samples
    vs = np.random.choice(vein, n_sub, replace=False)
    as_ = np.random.choice(artery, n_sub, replace=False)
    ps = np.sqrt(((n_sub-1)*np.var(vs,ddof=1)+(n_sub-1)*np.var(as_,ddof=1))/(2*n_sub-2))
    d_sub = (vs.mean()-as_.mean())/ps
    # quick bootstrap CI
    bd = []
    for _ in range(500):
        bv = np.random.choice(vs, n_sub, replace=True)
        ba = np.random.choice(as_, n_sub, replace=True)
        p2 = np.sqrt(((n_sub-1)*np.var(bv,ddof=1)+(n_sub-1)*np.var(ba,ddof=1))/(2*n_sub-2))
        bd.append((bv.mean()-ba.mean())/p2)
    lo, hi = np.percentile(bd, [2.5, 97.5])
    print(f"  n={n_sub:2d}/pack  d={d_sub:6.3f}  95% CI [{lo:6.3f}, {hi:6.3f}]  "
          f"{'crosses 0' if lo<0<hi else 'excludes 0'}")

print("\n=== Variability sensitivity (mean difference fixed) ===")
raw_diff = vein.mean() - artery.mean()
for scale in [0.7, 1.0, 1.5, 2.0]:
    # scale the residuals around each group mean
    v2 = vein.mean() + (vein - vein.mean()) * scale
    a2 = artery.mean() + (artery - artery.mean()) * scale
    ps = np.sqrt(((nx-1)*np.var(v2,ddof=1)+(ny-1)*np.var(a2,ddof=1))/(nx+ny-2))
    d2 = raw_diff / ps   # same numerator, larger denominator when scale>1
    print(f"  SD scale ×{scale:.1f}  →  d = {d2:.3f}  ({classify_d(d2)})")


## 7. Alternate code paths

### 7.1 Hedges’ g (small-sample correction to d)


In [ ]:
# Hedges' g = d * J, where J ≈ 1 - 3/(4*df-1)
df = nx + ny - 2
J = 1 - 3/(4*df - 1)
g = d * J
print(f"Cohen's d = {d:.4f}")
print(f"Hedges' g = {g:.4f}  (correction factor J = {J:.4f})")
print("With n=20 per group the correction is tiny; still good practice to report g for small samples.")


### 7.2 η² from statsmodels ANOVA table (if available)


In [ ]:
try:
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    model = smf.ols("score ~ C(pack)", data=iron).fit()
    anova_tbl = sm.stats.anova_lm(model, typ=2)
    ss_b = anova_tbl.loc["C(pack)", "sum_sq"]
    ss_r = anova_tbl.loc["Residual", "sum_sq"]
    eta2_sm = ss_b / (ss_b + ss_r)
    print(anova_tbl)
    print(f"\nη² from statsmodels = {eta2_sm:.4f}  (matches manual {eta2:.4f})")
except ImportError:
    print("statsmodels not installed – skipping.")


## 8. Business Implications & Mini Report Draft

**Problem**  
A statistically significant p-value tells Familiar that a difference exists; an effect size tells them *how large* the difference is and whether it is worth building a marketing claim or a clinical-counselling protocol around.

**Magnitudes (with plain-language glosses)**
- Lifespan: Cohen’s *d* ≈ 0.62 (medium).  Vein subscribers live roughly 1.3 years longer on average; the bootstrap CI is wide and still includes values below the conventional “medium” threshold, so the claim should be modest.  
- Iron (categorical): Cramér’s *V* ≈ 0.57 (large).  A 50-percentage-point gap in low-iron prevalence is clinically obvious and immediately actionable.  
- Iron (ordinal): η² ≈ 0.32 (large).  Pack membership explains about one-third of the variance in the ordinal iron score.

**Recommendation**
- Marketing may cite a longevity benefit for the Vein Pack, but should avoid over-stating precision until a larger sample tightens the CI for *d*.  
- Side-effect counselling can be differentiated with high confidence: Vein → iron-deficiency monitoring; Artery → iron-overload monitoring.

**Caution**  
With only 20 observations per lifespan arm the CI for Cohen’s *d* is wide.  The iron results (n = 345) are far more precise.  Always present the effect size together with its uncertainty and a concrete unit (years, percentage points) that non-technical stakeholders can act on.


## End of Solution Notebook

You now have a complete effect-size interpretation toolkit that sits on top of the earlier Familiar analyses.  Re-run the sensitivity cells with different sub-sample sizes or SD scales to see how classification boundaries shift.
